# Figure 1B — Performance Metrics Matrix

9-row performance matrix (FN, FP, TP, TN, Sensitivity, Specificity, Precision, F1, AUC) across 6 toxicities. FN/FP cells are split-colored by model error (purple) vs gold standard curation error (red).

**Data sources:**
- `llama_maverick_1k_results.csv` — model predictions
- `2026May01_merged_ae_with_apr_full.csv` — original (uncorrected) gold standard
- `final_gold_standard_1k.csv` — corrected gold standard
- `train_mrns.csv` (optional, for reproducible split)
- `test_mrns.csv` (optional, for reproducible split)


In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix, accuracy_score
from sklearn.metrics import fbeta_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

%matplotlib inline


In [ ]:
# ---------------------------------------------------------------------------
# FILE PATHS
# ---------------------------------------------------------------------------
ROOT = Path("..").resolve()
FIGURES = ROOT.parent.parent
DATA = FIGURES / "figures_data" / "figure 1" / "data"
RESULTS = ROOT / "results"
(RESULTS / "main").mkdir(parents=True, exist_ok=True)
(RESULTS / "supp").mkdir(parents=True, exist_ok=True)
print(f"Data: {DATA}")

MODEL_FILE = DATA / "llama_maverick_1k_results.csv"
ORIGINAL_GOLD = DATA / "2026May01_merged_ae_with_apr_full.csv"
CORRECTED_GOLD = DATA / "final_gold_standard_1k.csv"
SPLIT_DIR = DATA
OUT_PATH = RESULTS / "main" / "Performance_Matrix_Fig1B.pdf"

for p in [MODEL_FILE, ORIGINAL_GOLD, CORRECTED_GOLD]:
    assert p.exists(), f"Missing: {p.name}"

# Check for optional split files
HAS_SPLIT = (SPLIT_DIR / "train_mrns.csv").exists() and (SPLIT_DIR / "test_mrns.csv").exists()
print(f"All required files found. Split files: {'yes' if HAS_SPLIT else 'no (will regenerate)'}")


In [ ]:
# ---------------------------------------------------------------------------
# CONSTANTS
# ---------------------------------------------------------------------------
TOXICITIES = [
    "liver_toxicity", "hypothyroidism", "pneumonitis",
    "colitis", "adrenal_insufficiency", "hyperthyroidism",
]

OPTIMAL_THRESHOLDS = {
    'pneumonitis': 0.710,
    'adrenal_insufficiency': 0.810,
    'liver_toxicity': 0.010,
    'colitis': 0.710,
    'hyperthyroidism': 0.810,
    'hypothyroidism': 0.510,
}

DISPLAY_ORDER = [
    "Liver\nToxicity", "Hypo-\nthyroidism", "Pneumonitis",
    "Colitis", "Adrenal\nInsufficiency", "Hyper-\nthyroidism",
]

TOXICITY_MAP = {
    'pneumonitis': 'pneumonitis',
    'adrenal insufficiency': 'adrenal_insufficiency',
    'adrenal_insufficiency': 'adrenal_insufficiency',
    'liver toxicity': 'liver_toxicity',
    'liver_toxicity': 'liver_toxicity',
    'colitis': 'colitis',
    'hyperthyroidism': 'hyperthyroidism',
    'hypothyroidism': 'hypothyroidism',
}


In [ ]:
# ---------------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------------
def _read_csv(path):
    try:
        return pd.read_csv(path, low_memory=False)
    except UnicodeDecodeError:
        for enc in ("latin-1", "cp1252"):
            try:
                return pd.read_csv(path, encoding=enc, low_memory=False)
            except UnicodeDecodeError:
                continue
        raise

def _normalize_mrn_list(mrns):
    normalized = []
    for value in mrns:
        if pd.isna(value): continue
        normalized.append(str(value).strip().lstrip("\'").zfill(8))
    return normalized

def calculate_metrics_at_threshold(y_true, y_score, threshold):
    y_pred = (y_score >= (threshold - 1e-10)).astype(int)
    if len(np.unique(y_true)) < 2:
        if np.all(y_true == 0):
            tn, fp = int(np.sum(y_pred == 0)), int(np.sum(y_pred == 1))
            tp = fn = 0
        else:
            tp, fn = int(np.sum(y_pred == 1)), int(np.sum(y_pred == 0))
            tn = fp = 0
    else:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    specificity = tn / (tn + fp) if (tn + fp) else 0
    return {"TP": tp, "FP": fp, "TN": tn, "FN": fn,
            "Precision": precision, "Recall": recall, "F1_Score": f1,
            "Specificity": specificity, "Threshold": threshold}


In [ ]:
# ---------------------------------------------------------------------------
# Load MRN of manually-added patient (kept out of version control -- contains PHI)
# ---------------------------------------------------------------------------
NEW_MRN_FILE = SPLIT_DIR / "new_patient_mrn.txt"  # REMINDER: not tracked in git -- add to .gitignore
if not NEW_MRN_FILE.exists():
    raise FileNotFoundError(
        f"{NEW_MRN_FILE} not found. Create this file (outside version control) "
        "containing only the zero-padded MRN of the manually added patient."
    )
NEW_MRN = NEW_MRN_FILE.read_text().strip().zfill(8)

In [ ]:
# ---------------------------------------------------------------------------
# Load model predictions + original gold standard (long format)
# ---------------------------------------------------------------------------
print("Loading model predictions...")
df_model = _read_csv(MODEL_FILE)
df_model = df_model.rename(columns={
    "liver toxicity": "liver_toxicity",
    "adrenal insufficiency": "adrenal_insufficiency",
    "doc": "cdd_doc_guid",
})
df_model["mrn"] = df_model["mrn"].astype(str).str.zfill(8)

print("Loading original gold standard...")
df_gold = _read_csv(ORIGINAL_GOLD)
df_gold["MRN"] = df_gold["MRN"].astype(str).str.zfill(8)

# Add missing patient found in another source, with liver toxicity
new_row = {col: np.nan for col in df_gold.columns}
new_row["MRN"] = NEW_MRN
new_row["Toxicity"] = "liver toxicity"
df_gold = pd.concat([df_gold, pd.DataFrame([new_row])], ignore_index=True)

gold_mrns = set(df_gold["MRN"])

model_mrns = set(df_model["mrn"])
gold_mrns = set(df_gold["MRN"])
shared_mrns = model_mrns & gold_mrns

# Build original patient truth from long-format gold standard
patient_truth_original = {mrn: {tox: 0 for tox in TOXICITIES} for mrn in shared_mrns}
for _, row in df_gold[df_gold["MRN"].isin(shared_mrns)].iterrows():
    mapped = TOXICITY_MAP.get(row["Toxicity"].strip().lower())
    if mapped and mapped in TOXICITIES:
        patient_truth_original[row["MRN"]][mapped] = 1
patient_truth_original = pd.DataFrame.from_dict(patient_truth_original, orient="index").sort_index().astype(int)

# Patient-level max scores
df_model_filtered = df_model[df_model["mrn"].isin(shared_mrns)]
patient_scores = df_model_filtered.groupby("mrn")[TOXICITIES].max().sort_index()

print(f"Model: {len(df_model):,} rows, {len(model_mrns):,} patients")
print(f"Original GS: {len(df_gold):,} rows, {len(gold_mrns):,} patients")
print(f"Shared: {len(shared_mrns):,} patients")


In [ ]:
# ---------------------------------------------------------------------------
# Load corrected gold standard
# ---------------------------------------------------------------------------
print("Loading corrected gold standard...")
df_corrected = _read_csv(CORRECTED_GOLD)
mrn_col = next((c for c in ["MRN", "mrn", "MRN_STR"] if c in df_corrected.columns), None)
df_corrected["MRN"] = df_corrected[mrn_col].astype(str).str.lstrip("'").str.zfill(8)
df_corrected = df_corrected.set_index("MRN")

patient_truth_corrected = pd.DataFrame(index=sorted(shared_mrns), columns=TOXICITIES)
for mrn in patient_truth_corrected.index:
    for tox in TOXICITIES:
        if mrn in df_corrected.index and tox in df_corrected.columns:
            patient_truth_corrected.at[mrn, tox] = int(pd.to_numeric(df_corrected.at[mrn, tox], errors="coerce") or 0)
        else:
            patient_truth_corrected.at[mrn, tox] = 0
patient_truth_corrected = patient_truth_corrected.astype(int).sort_index()

# Align all on common MRNs
common_mrns = set(patient_scores.index) & set(patient_truth_original.index) & set(patient_truth_corrected.index)
patient_scores = patient_scores.loc[sorted(common_mrns)]
patient_truth_original = patient_truth_original.loc[sorted(common_mrns)]
patient_truth_corrected = patient_truth_corrected.loc[sorted(common_mrns)]

print(f"Corrected GS: {len(df_corrected):,} patients")
print(f"Common MRNs: {len(common_mrns):,}")


In [ ]:
# ---------------------------------------------------------------------------
# Train/test split
# ---------------------------------------------------------------------------
NEW_MRN_FILE = SPLIT_DIR / "new_patient_mrn.txt"  # REMINDER: not tracked in git -- add to .gitignore
if not NEW_MRN_FILE.exists():
    raise FileNotFoundError(
        f"{NEW_MRN_FILE} not found. Create this file (outside version control) "
        "containing only the zero-padded MRN of the manually added patient."
    )
NEW_MRN = NEW_MRN_FILE.read_text().strip().zfill(8)

if HAS_SPLIT:
    print("Loading saved train/test split...")
    train_df = pd.read_csv(SPLIT_DIR / "train_mrns.csv")
    test_df = pd.read_csv(SPLIT_DIR / "test_mrns.csv")
    train_col = next(c for c in train_df.columns if c.lower() in {'mrn', 'mrn_str'})
    test_col = next(c for c in test_df.columns if c.lower() in {'mrn', 'mrn_str'})
    train_indices = _normalize_mrn_list(train_df[train_col].tolist())
    test_indices = _normalize_mrn_list(test_df[test_col].tolist())
    available = set(patient_scores.index)
    train_indices = [m for m in train_indices if m in available]
    test_indices = [m for m in test_indices if m in available]
else:
    print("Regenerating original split (on pool WITHOUT the newly added patient)...")
    np.random.seed(3334)
    all_indices = [m for m in patient_scores.index.tolist() if m != NEW_MRN]
    train_indices, test_indices = train_test_split(all_indices, train_size=700, random_state=42, shuffle=True)

    # Add the new patient to TEST -- train stays 700, test becomes 297 + 1 = 298
    if NEW_MRN not in patient_scores.index:
        raise ValueError(
            f"NEW_MRN not found in patient_scores.index (len={len(NEW_MRN)}). "
            "Check new_patient_mrn.txt matches the zfill(8) format used elsewhere."
        )
    if NEW_MRN not in train_indices and NEW_MRN not in test_indices:
        test_indices.append(NEW_MRN)

    # Lock this split so it never regenerates/reshuffles again
    pd.DataFrame({"MRN": train_indices}).to_csv(SPLIT_DIR / "train_mrns.csv", index=False)
    pd.DataFrame({"MRN": test_indices}).to_csv(SPLIT_DIR / "test_mrns.csv", index=False)
    print("Saved split to train_mrns.csv and test_mrns.csv")

print(f"Train: {len(train_indices)}, Test: {len(test_indices)}")

test_scores = patient_scores.loc[test_indices]
test_truth_original = patient_truth_original.loc[test_indices]
test_truth_corrected = patient_truth_corrected.loc[test_indices]

In [ ]:
ADDITIONAL_NEG_MRN_FILE = SPLIT_DIR / "additional_negative_mrns.txt"  # not tracked in git -- one MRN per line
ADDITIONAL_NEG_MRNS = [
    line.strip().zfill(8) for line in ADDITIONAL_NEG_MRN_FILE.read_text().splitlines() if line.strip()
]
assert len(ADDITIONAL_NEG_MRNS) == 2, f"Expected 2 MRNs, got {len(ADDITIONAL_NEG_MRNS)}"

# These 2 patients were filtered out by the RAG retrieval stage and never reached
# the LLM classifier. This IS the pipeline's real behavior: no retrieved evidence ->
# no LLM call -> negative prediction for every toxicity. A score of 0.0 reflects
# this actual pipeline output, not a missing value.
for mrn in ADDITIONAL_NEG_MRNS:
    if mrn in test_scores.index:
        continue
    zero_row = pd.DataFrame([[0.0] * len(TOXICITIES)], columns=TOXICITIES, index=[mrn])
    test_scores = pd.concat([test_scores, zero_row])
    test_truth_original = pd.concat([test_truth_original, zero_row.astype(int)])
    test_truth_corrected = pd.concat([test_truth_corrected, zero_row.astype(int)])

print(f"Test set size after additions: {len(test_scores)}")

In [ ]:
# ---------------------------------------------------------------------------
# Compute all metrics
# ---------------------------------------------------------------------------
fn_results, fp_results = {}, {}
real_positives, real_negatives = {}, {}
auc_values, recall_values, precision_values, specificity_values = {}, {}, {}, {}

for tox in TOXICITIES:
    thr = OPTIMAL_THRESHOLDS[tox]
    y_score = test_scores[tox].astype(float)
    y_pred = (y_score >= (thr - 1e-10)).astype(int)
    y_orig = test_truth_original[tox]
    y_corr = test_truth_corrected[tox]

    # FN: predicted 0, original GS says 1
    fn_mask = (y_pred == 0) & (y_orig == 1)
    fn_idx = test_scores.index[fn_mask].tolist()
    fn_model = sum(1 for m in fn_idx if y_corr.loc[m] == 1)
    fn_curation = sum(1 for m in fn_idx if y_corr.loc[m] == 0)
    fn_results[tox] = {"total": len(fn_idx), "model_error": fn_model, "curation_error": fn_curation}

    # FP: predicted 1, original GS says 0
    fp_mask = (y_pred == 1) & (y_orig == 0)
    fp_idx = test_scores.index[fp_mask].tolist()
    fp_model = sum(1 for m in fp_idx if y_corr.loc[m] == 0)
    fp_curation = sum(1 for m in fp_idx if y_corr.loc[m] == 1)
    fp_results[tox] = {"total": len(fp_idx), "model_error": fp_model, "curation_error": fp_curation}

    # TP/TN on ORIGINAL GS -- same basis as FN/FP, so FN+FP+TP+TN == test set size
    real_positives[tox] = int(((y_pred == 1) & (y_orig == 1)).sum())
    real_negatives[tox] = int(((y_pred == 0) & (y_orig == 0)).sum())

    # AUC -- on corrected GS
    try:
        if y_corr.sum() > 0 and (y_corr == 0).sum() > 0:
            fpr, tpr, _ = roc_curve(y_corr, y_score)
            auc_values[tox] = auc(fpr, tpr)
        else:
            auc_values[tox] = np.nan
    except:
        auc_values[tox] = np.nan

    # Recall, Precision & Specificity -- all on corrected GS (single consistent call)
    m = calculate_metrics_at_threshold(y_corr, y_score, thr)
    recall_values[tox] = m["Recall"]
    precision_values[tox] = m["Precision"]
    specificity_values[tox] = m["Specificity"]

    assert fn_results[tox]["total"] + fp_results[tox]["total"] + real_positives[tox] + real_negatives[tox] == len(test_scores), \
        f"{tox}: FN+FP+TP+TN should equal test set size ({len(test_scores)})"

    print(f"{tox}: FN={fn_results[tox]['total']} (model={fn_model}, curation={fn_curation}), "
          f"FP={fp_results[tox]['total']} (model={fp_model}, curation={fp_curation}), "
          f"TP={real_positives[tox]}, TN={real_negatives[tox]}, "
          f"AUC={auc_values[tox]:.2f}, Recall={recall_values[tox]:.2f}, Prec={precision_values[tox]:.2f}, "
          f"Spec={specificity_values[tox]:.2f}")


In [ ]:
# ---------------------------------------------------------------------------
# Build summary CSV (intermediate, used by plotting cell)
# ---------------------------------------------------------------------------
summary_rows = []
for tox in TOXICITIES:
    fn = fn_results[tox]
    fp = fp_results[tox]
    summary_rows.append({
        "Toxicity": tox.replace("_", " ").title(),
        "FN_Total": fn["total"], "FN_Model_Error": fn["model_error"], "FN_Curation_Error": fn["curation_error"],
        "FP_Total": fp["total"], "FP_Model_Error": fp["model_error"], "FP_Curation_Error": fp["curation_error"],
        "Real_Positive": real_positives[tox], "Real_Negative": real_negatives[tox],
        "AUC": auc_values[tox], "Recall": recall_values[tox], "Precision": precision_values[tox],
        "Specificity": specificity_values[tox],
    })

summary_df = pd.DataFrame(summary_rows)

assert (summary_df["FN_Total"] + summary_df["FP_Total"] + summary_df["Real_Positive"] + summary_df["Real_Negative"] == len(test_scores)).all(), \
    "FN+FP+TP+TN does not equal test set size for at least one toxicity"

In [ ]:
# ---------------------------------------------------------------------------
# Figure — Performance Matrix
# ---------------------------------------------------------------------------
order = ["Liver Toxicity", "Hypothyroidism", "Pneumonitis",
         "Colitis", "Adrenal Insufficiency", "Hyperthyroidism"]
df = summary_df.copy()
df["Toxicity"] = df["Toxicity"].str.title()
df = df[df["Toxicity"].isin(order)].set_index("Toxicity").loc[order]

row_labels = ["False Negatives", "False Positives", "True Positives", "True Negatives",
              "Sensitivity", "Specificity", "Precision", "F1 Score", "AUC"]

n_rows, n_cols = len(row_labels), len(order)

# Build data matrix
data_matrix = np.zeros((n_rows, n_cols), dtype=object)
fn_vals = df["FN_Total"].to_numpy()
fp_vals = df["FP_Total"].to_numpy()
tp_vals = df["Real_Positive"].to_numpy()
tn_vals = df["Real_Negative"].to_numpy()

data_matrix[0] = fn_vals
data_matrix[1] = fp_vals
data_matrix[2] = tp_vals
data_matrix[3] = tn_vals
data_matrix[4] = df["Recall"].map(lambda x: f"{x:.2f}" if not np.isnan(x) else "N/A").to_numpy()
data_matrix[5] = df["Specificity"].map(lambda x: f"{x:.2f}" if not np.isnan(x) else "N/A").to_numpy()
data_matrix[6] = df["Precision"].map(lambda x: f"{x:.2f}" if not np.isnan(x) else "N/A").to_numpy()

f1_vals = []
for p, r in zip(df["Precision"], df["Recall"]):
    f1_vals.append(f"{2*p*r/(p+r):.2f}" if (p+r) > 0 else "N/A")
data_matrix[7] = np.array(f1_vals, dtype=object)
data_matrix[8] = df["AUC"].map(lambda x: f"{x:.2f}" if not np.isnan(x) else "N/A").to_numpy()

fn_model = df["FN_Model_Error"].to_numpy()
fn_curation = df["FN_Curation_Error"].to_numpy()
fp_model = df["FP_Model_Error"].to_numpy()
fp_curation = df["FP_Curation_Error"].to_numpy()

color_map = {
    "False Negatives": "#d7e3fe", "False Positives": "#f8c7c3",
    "True Positives": "#fff2b2", "True Negatives": "#fff2b2",
    "Sensitivity": "#d7e3fe", "Specificity": "#d7e3fe",
    "Precision": "#d7e3fe", "F1 Score": "#d7e3fe", "AUC": "#d7e3fe",
}

plt.rcParams.update({
    "font.family": "Arial",
    "pdf.fonttype": 42, "ps.fonttype": 42,
})
fig, ax = plt.subplots(figsize=(4.05, 2.7), constrained_layout=True)
ax.set_xlim(0, n_cols)
ax.set_ylim(0, n_rows + 1.6)
ax.axis("off")

cell_w, cell_h = 1, 1

for r in range(n_rows):
    for c in range(n_cols):
        x, y = c * cell_w, n_rows - r - 1
        label = row_labels[r]
        base_color = color_map[label]

        if label in {"False Negatives", "False Positives"}:
            total = data_matrix[r, c]
            if total == 0:
                ax.add_patch(plt.Rectangle((x, y), cell_w, cell_h, facecolor=base_color, edgecolor="white"))
            else:
                if label == "False Negatives":
                    mv, cv = fn_model[c], fn_curation[c]
                else:
                    mv, cv = fp_model[c], fp_curation[c]
                mr, cr = mv/total, cv/total
                start = x
                if mr > 0:
                    ax.add_patch(plt.Rectangle((start, y), mr*cell_w, cell_h, facecolor="#dcc6ec", edgecolor="white"))
                    start += mr * cell_w
                if cr > 0:
                    ax.add_patch(plt.Rectangle((start, y), cr*cell_w, cell_h, facecolor="#f8c7c3", edgecolor="white"))
                    start += cr * cell_w
                if start < x + cell_w:
                    ax.add_patch(plt.Rectangle((start, y), x+cell_w-start, cell_h, facecolor=base_color, edgecolor="white"))
        else:
            ax.add_patch(plt.Rectangle((x, y), cell_w, cell_h, facecolor=base_color, edgecolor="white"))

        ax.text(x + 0.5, y + 0.5, str(data_matrix[r, c]), ha="center", va="center", fontsize=5, fontweight="bold")

# Column headers
for c, label in enumerate(DISPLAY_ORDER):
    ax.text(c + 0.5, n_rows + 0.35, label, ha="center", va="bottom", fontsize=5, fontweight="bold")

# Row labels
for r, label in enumerate(row_labels):
    ax.text(-0.1, n_rows - r - 1 + 0.5, label, ha="right", va="center", fontsize=6, fontweight="bold")

# Legend
legend_y = n_rows + 1.15
ax.add_patch(plt.Rectangle((0.0, legend_y), 0.3, 0.3, facecolor="#dcc6ec", edgecolor="white"))
ax.text(0.4, legend_y + 0.15, "Model error", ha="left", va="center", fontsize=5, fontweight="bold")
ax.add_patch(plt.Rectangle((2.0, legend_y), 0.3, 0.3, facecolor="#f8c7c3", edgecolor="white"))
ax.text(2.4, legend_y + 0.15, "Gold standard error", ha="left", va="center", fontsize=5, fontweight="bold")

fig.savefig(str(OUT_PATH), format="pdf")
print(f"Saved: {OUT_PATH.name}")
plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# 95% Confidence Intervals via patient-level bootstrap
# ---------------------------------------------------------------------------
N_BOOTSTRAP = 2000
rng = np.random.default_rng(42)

ci_rows = []

for tox in TOXICITIES:
    thr = OPTIMAL_THRESHOLDS[tox]
    y_score_full = test_scores[tox].astype(float)
    y_corr_full = test_truth_corrected[tox]
    n = len(y_score_full)

    boot_sens, boot_spec, boot_prec, boot_f1, boot_auc = [], [], [], [], []

    for _ in range(N_BOOTSTRAP):
        idx = rng.integers(0, n, size=n)  # sample patient indices with replacement
        y_score_b = y_score_full.iloc[idx].reset_index(drop=True)
        y_corr_b = y_corr_full.iloc[idx].reset_index(drop=True)
        y_pred_b = (y_score_b >= (thr - 1e-10)).astype(int)

        tp = int(((y_pred_b == 1) & (y_corr_b == 1)).sum())
        fp = int(((y_pred_b == 1) & (y_corr_b == 0)).sum())
        tn = int(((y_pred_b == 0) & (y_corr_b == 0)).sum())
        fn = int(((y_pred_b == 0) & (y_corr_b == 1)).sum())

        sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        prec = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        f1 = 2 * prec * sens / (prec + sens) if (prec and sens and (prec + sens) > 0) else np.nan

        boot_sens.append(sens)
        boot_spec.append(spec)
        boot_prec.append(prec)
        boot_f1.append(f1)

        # AUC needs both classes present in the resample
        try:
            if y_corr_b.sum() > 0 and (y_corr_b == 0).sum() > 0:
                fpr, tpr, _ = roc_curve(y_corr_b, y_score_b)
                boot_auc.append(auc(fpr, tpr))
            else:
                boot_auc.append(np.nan)
        except Exception:
            boot_auc.append(np.nan)

    def _ci(vals, point_estimate):
        vals = np.array([v for v in vals if not np.isnan(v)])
        if len(vals) == 0:
            return point_estimate, np.nan, np.nan
        lo, hi = np.percentile(vals, [2.5, 97.5])
        return point_estimate, lo, hi

    metrics_point = {
        "Sensitivity": recall_values[tox],
        "Specificity": specificity_values[tox],
        "Precision": precision_values[tox],
        "F1": (2 * precision_values[tox] * recall_values[tox] / (precision_values[tox] + recall_values[tox])
               if (precision_values[tox] + recall_values[tox]) > 0 else np.nan),
        "AUC": auc_values[tox],
    }
    boot_dists = {
        "Sensitivity": boot_sens, "Specificity": boot_spec,
        "Precision": boot_prec, "F1": boot_f1, "AUC": boot_auc,
    }

    for metric_name, point in metrics_point.items():
        est, lo, hi = _ci(boot_dists[metric_name], point)
        ci_rows.append({
            "Toxicity": tox.replace("_", " ").title(),
            "Metric": metric_name,
            "Estimate": round(est, 3) if not np.isnan(est) else np.nan,
            "CI_Lower": round(lo, 3) if not np.isnan(lo) else np.nan,
            "CI_Upper": round(hi, 3) if not np.isnan(hi) else np.nan,
            "N_Bootstrap": N_BOOTSTRAP,
        })

    print(f"{tox}: Sens={metrics_point['Sensitivity']:.2f} "
          f"[{ci_rows[-5]['CI_Lower']:.2f}-{ci_rows[-5]['CI_Upper']:.2f}], "
          f"AUC={metrics_point['AUC']:.2f} [{ci_rows[-1]['CI_Lower']:.2f}-{ci_rows[-1]['CI_Upper']:.2f}]")

ci_df = pd.DataFrame(ci_rows)
CI_OUT_PATH = OUT_PATH.parent / "figure1b_confidence_intervals.csv"
ci_df.to_csv(CI_OUT_PATH, index=False)
print(f"\nSaved: {CI_OUT_PATH.name}")

In [ ]:
# ===========================================================================
# SUPPLEMENTARY TABLE — held-out validation set (GS patients outside the 1k)
#
# Same thresholds as Fig 1B, applied to every gold-standard patient NOT in the
# locked train/test split. Scores come from the 84k run.
# ===========================================================================

VALIDATION_MODEL_FILE = DATA / "llama_maverick_84k_patient_results.csv"
VAL_CSV_OUT = RESULTS / "supp" / "Validation_Set_Performance_SuppTable.csv"
assert VALIDATION_MODEL_FILE.exists(), f"Missing: {VALIDATION_MODEL_FILE.name}"

# ---- 1. Every patient in the original GS ----------------------------------
gs_all_mrns = set(df_gold["MRN"].dropna())

# ---- 2. Remove the locked train/test cohort -------------------------------
in_1k = set(train_indices) | set(test_indices) | set(ADDITIONAL_NEG_MRNS) | {NEW_MRN}
validation_mrns = gs_all_mrns - in_1k
print(f"GS patients total:        {len(gs_all_mrns):,}")
print(f"In locked 1k split:       {len(in_1k & gs_all_mrns):,}")
print(f"Validation candidates:    {len(validation_mrns):,}")

# ---- 3. Model scores from the 84k run -------------------------------------
df_val_model = _read_csv(VALIDATION_MODEL_FILE)
df_val_model = df_val_model.rename(columns={
    "liver toxicity": "liver_toxicity",
    "adrenal insufficiency": "adrenal_insufficiency",
})
df_val_model["mrn"] = df_val_model["mrn"].astype(str).str.lstrip("'").str.zfill(8)
for tox in TOXICITIES:
    df_val_model[tox] = pd.to_numeric(df_val_model[tox], errors="coerce").fillna(0.0)

scored_mrns = set(df_val_model["mrn"])
val_mrns = sorted(validation_mrns & scored_mrns)
print(f"Of which scored in 84k:   {len(val_mrns):,}"
      f"  ({len(validation_mrns - scored_mrns):,} GS patients have no 84k score)")

val_scores = (df_val_model[df_val_model["mrn"].isin(val_mrns)]
              .groupby("mrn")[TOXICITIES].max().sort_index())

# ---- 4. Truth from the ORIGINAL GS (long format; absent row = negative) ----
val_truth = pd.DataFrame(0, index=val_scores.index, columns=TOXICITIES, dtype=int)
_gs_val = df_gold[df_gold["MRN"].isin(val_scores.index)]
for _, row in _gs_val.iterrows():
    mapped = TOXICITY_MAP.get(str(row["Toxicity"]).strip().lower())
    if mapped in TOXICITIES:
        val_truth.at[row["MRN"], mapped] = 1

assert val_scores.index.equals(val_truth.index), "score/truth index mismatch"
print(f"Validation set: {len(val_scores):,} patients\n")

# ---- 5. Metrics + patient-level bootstrap CIs -----------------------------
N_BOOTSTRAP_VAL = 2000
rng_val = np.random.default_rng(42)
val_rows = []

for tox in TOXICITIES:
    thr = OPTIMAL_THRESHOLDS[tox]
    y_score = val_scores[tox].astype(float).to_numpy()
    y_true = val_truth[tox].to_numpy()
    n = len(y_score)

    m = calculate_metrics_at_threshold(y_true, y_score, thr)
    try:
        fpr, tpr, _ = roc_curve(y_true, y_score)
        auc_val = auc(fpr, tpr) if (y_true.sum() and (y_true == 0).sum()) else np.nan
    except Exception:
        auc_val = np.nan

    b_sens, b_spec, b_prec, b_f1, b_auc = [], [], [], [], []
    for _ in range(N_BOOTSTRAP_VAL):
        idx = rng_val.integers(0, n, size=n)
        yt, ys = y_true[idx], y_score[idx]
        yp = (ys >= (thr - 1e-10)).astype(int)
        tp = int(((yp == 1) & (yt == 1)).sum()); fp = int(((yp == 1) & (yt == 0)).sum())
        tn = int(((yp == 0) & (yt == 0)).sum()); fn = int(((yp == 0) & (yt == 1)).sum())
        se = tp / (tp + fn) if (tp + fn) else np.nan
        sp = tn / (tn + fp) if (tn + fp) else np.nan
        pr = tp / (tp + fp) if (tp + fp) else np.nan
        b_sens.append(se); b_spec.append(sp); b_prec.append(pr)
        b_f1.append(2 * pr * se / (pr + se) if (pr and se and (pr + se)) else np.nan)
        try:
            if yt.sum() and (yt == 0).sum():
                f_, t_, _ = roc_curve(yt, ys); b_auc.append(auc(f_, t_))
            else:
                b_auc.append(np.nan)
        except Exception:
            b_auc.append(np.nan)

    def _ci(v):
        v = np.asarray([x for x in v if not np.isnan(x)])
        return (np.percentile(v, 2.5), np.percentile(v, 97.5)) if len(v) else (np.nan, np.nan)

    f1_pt = (2 * m["Precision"] * m["Recall"] / (m["Precision"] + m["Recall"])
             if (m["Precision"] + m["Recall"]) else np.nan)
    se_lo, se_hi = _ci(b_sens); sp_lo, sp_hi = _ci(b_spec)
    pr_lo, pr_hi = _ci(b_prec); f1_lo, f1_hi = _ci(b_f1); au_lo, au_hi = _ci(b_auc)

    val_rows.append({
        "toxicity": tox.replace("_", " ").title(), "threshold": thr,
        "n_patients": n, "n_positive": int(y_true.sum()),
        "TP": m["TP"], "FP": m["FP"], "TN": m["TN"], "FN": m["FN"],
        "sensitivity": m["Recall"], "sensitivity_ci_low": se_lo, "sensitivity_ci_high": se_hi,
        "specificity": m["Specificity"], "specificity_ci_low": sp_lo, "specificity_ci_high": sp_hi,
        "precision": m["Precision"], "precision_ci_low": pr_lo, "precision_ci_high": pr_hi,
        "f1": f1_pt, "f1_ci_low": f1_lo, "f1_ci_high": f1_hi,
        "auc": auc_val, "auc_ci_low": au_lo, "auc_ci_high": au_hi,
        "label_basis": "original_gold_standard",
    })
    print(f"  {tox:<24} Sens={m['Recall']:.2f} Spec={m['Specificity']:.2f} "
          f"Prec={m['Precision']:.2f} F1={f1_pt:.2f} AUC={auc_val:.2f}  "
          f"(n_pos={int(y_true.sum()):,})")

validation_df = pd.DataFrame(val_rows)
assert not any(c.lower() in ("mrn", "mrn_std") for c in validation_df.columns), "MRN leaked"
VAL_CSV_OUT.parent.mkdir(parents=True, exist_ok=True)
validation_df.to_csv(VAL_CSV_OUT, index=False)
print(f"\nSaved: {VAL_CSV_OUT.name}")

In [ ]:
missing_from_84k = sorted(validation_mrns - scored_mrns)

print(f"GS validation candidates: {len(validation_mrns):,}")
print(f"Found in 84k results:      {len(validation_mrns & scored_mrns):,}")
print(f"Missing from 84k results:  {len(missing_from_84k):,}")

# How many of the missing patients actually have a GS annotation?
missing_gs = df_gold[df_gold["MRN"].isin(missing_from_84k)]

print(f"Missing patients with GS rows: {missing_gs['MRN'].nunique():,}")
print(f"Missing patients with toxicities: "
      f"{missing_gs.loc[missing_gs['Toxicity'].notna(), 'MRN'].nunique():,}")

print("\nGS toxicity counts among missing patients:")
print(missing_gs["Toxicity"].value_counts(dropna=False))

In [ ]:
df_84k = pd.read_csv(VALIDATION_MODEL_FILE)

print(f"Rows in 84k file:       {len(df_84k):,}")
print(f"Unique patients in 84k: {df_84k['mrn'].nunique():,}")
print(f"Duplicate patient rows: {len(df_84k) - df_84k['mrn'].nunique():,}")

In [ ]:
gs_mrns = set(df_gold["MRN"].dropna().astype(str).str.zfill(8))
model_mrns = set(
    df_84k["mrn"]
    .dropna()
    .astype(str)
    .str.lstrip("'")
    .str.zfill(8)
)

print(f"GS unique patients:       {len(gs_mrns):,}")
print(f"84k unique patients:      {len(model_mrns):,}")
print(f"GS ∩ 84k:                 {len(gs_mrns & model_mrns):,}")
print(f"GS not in 84k:            {len(gs_mrns - model_mrns):,}")
print(f"84k not in GS:            {len(model_mrns - gs_mrns):,}")